In [ ]:
# ============================================================
# Cell 1 - Kaggle input tree
# Confirms which files are actually mounted, and under which
# layout. Paths are never hardcoded in this notebook.
# ============================================================

import os
import glob
import json
import numpy as np

ROOT = "/kaggle/input"
INTERESTING = (".csv", ".json", ".pth", ".pt")

print("=" * 70)
print("KAGGLE INPUT TREE")
print("=" * 70)

for dirpath, dirnames, filenames in os.walk(ROOT):
    depth = dirpath[len(ROOT):].count(os.sep)
    if depth > 4:
        dirnames[:] = []
        continue

    small = sorted(f for f in filenames if f.lower().endswith(INTERESTING))
    n_png = sum(1 for f in filenames if f.lower().endswith(".png"))

    if small or n_png or depth <= 2:
        pad = "  " * depth
        print(f"{pad}{os.path.basename(dirpath) or dirpath}/")
        for f in small[:15]:
            size = os.path.getsize(os.path.join(dirpath, f))
            print(f"{pad}  {f}  ({size:,} bytes)")
        if n_png:
            print(f"{pad}  [{n_png:,} png files]")

In [ ]:
# ============================================================
# Cell 2 - Locate the split and annotation files
# The Kaggle API will not list dataset files, so this is the
# check that settles whether localization data is present.
# ============================================================

TARGETS = [
    "train.csv",
    "val.csv",
    "test.csv",
    "loc_tune.csv",
    "loc_report.csv",
    "BBox_List_2017.csv",
    "split_metadata.json",
    "image_index.json",
]

found = {}

print("=" * 70)
print("SPLIT / ANNOTATION FILES")
print("=" * 70)

for name in TARGETS:
    hits = glob.glob(f"{ROOT}/**/{name}", recursive=True)
    found[name] = hits
    status = "FOUND  " if hits else "MISSING"
    location = hits[0] if hits else ""
    print(f"{name:<22} {status} {location}")

missing_loc = [
    n for n in ("loc_tune.csv", "loc_report.csv", "BBox_List_2017.csv")
    if not found[n]
]

print()
if missing_loc:
    print("NOTE: localization files not in the mounted dataset:", missing_loc)
    print("They exist in Google Drive per notebook 01 but may not have been")
    print("uploaded to the Kaggle splits dataset. IoBB work needs them.")
else:
    print("All localization inputs present.")

In [ ]:
# ============================================================
# Cell 3 - Resolve the frozen checkpoint
# ============================================================

ckpt_hits = glob.glob(f"{ROOT}/**/convnext_tiny_320_best.pth", recursive=True)
assert ckpt_hits, "convnext_tiny_320_best.pth not found under /kaggle/input"

CKPT_PATH = ckpt_hits[0]

thr_hits = glob.glob(
    f"{ROOT}/**/convnext_tiny_320_bce_thresholds.json", recursive=True
)

print("Checkpoint :", CKPT_PATH)
print("Size       : %.1f MB" % (os.path.getsize(CKPT_PATH) / 1e6))
print("Thresholds :", thr_hits[0] if thr_hits else "MISSING")

In [ ]:
# ============================================================
# Cell 4 - Build the architecture (must match training exactly)
# ============================================================

import torch
import torch.nn as nn
import torchvision
from torchvision.models import convnext_tiny

LABELS = [
    "Atelectasis",
    "Consolidation",
    "Infiltration",
    "Pneumothorax",
    "Edema",
    "Emphysema",
    "Fibrosis",
    "Effusion",
    "Pneumonia",
    "Pleural_Thickening",
    "Cardiomegaly",
    "Nodule",
    "Mass",
    "Hernia",
]

assert len(LABELS) == 14, "Project standard is 14 outputs"
assert "No_Finding" not in LABELS, "No_Finding is not a model output"

# machineShape is pinned to NvidiaTeslaT4 in the push payload (same as
# 01-convnext-training and resnet50-baseline). The default P100 pool this
# kernel got assigned earlier is sm_60, which the installed PyTorch cu128
# wheel does not support ("no kernel image is available"); T4 is sm_75,
# which is supported.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("torch       :", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu name      :", torch.cuda.get_device_name(0))
print("torchvision :", torchvision.__version__)
print("device      :", device)

# weights=None: ImageNet weights are overwritten by the checkpoint anyway
model = convnext_tiny(weights=None)
in_features = model.classifier[2].in_features
model.classifier[2] = nn.Linear(in_features, len(LABELS))

print("classifier in_features :", in_features)
print("output classes         :", len(LABELS))
print()
print(model.classifier)

In [ ]:
# ============================================================
# Cell 5 - Strict checkpoint load
# The ConvNeXt checkpoint is a dict, not a bare state_dict.
# ============================================================

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)

print("checkpoint top-level keys:", list(ckpt.keys()))
print()

result = model.load_state_dict(ckpt["model_state_dict"], strict=True)

print("missing keys    :", result.missing_keys)
print("unexpected keys :", result.unexpected_keys)

assert not result.missing_keys, "state_dict is missing keys"
assert not result.unexpected_keys, "state_dict has unexpected keys"

model = model.to(device)
model.eval()

print()
print("saved epoch     :", ckpt.get("epoch"))
print("val_macro_auc   :", ckpt.get("val_macro_auc"))
print()
print("STRICT LOAD OK - architecture matches the frozen checkpoint")

In [ ]:
# ============================================================
# Cell 6 - Real module structure (read, do not guess)
# ============================================================

print("len(model.features):", len(model.features))
print()

for i, block in enumerate(model.features):
    n_sub = len(block) if hasattr(block, "__len__") else 0
    print(f"features[{i}] : {type(block).__name__:<22} sub-modules={n_sub}")

print()
print("avgpool    :", type(model.avgpool).__name__)
print("classifier :", [type(m).__name__ for m in model.classifier])

In [ ]:
# ============================================================
# Cell 7 - Hook candidate Grad-CAM target layers
# features[7] is the last conv stage; features[5] is one stage
# earlier and gives a finer map if the last stage is too coarse.
# ============================================================

activations = {}


def capture(name):
    def hook(module, inputs, output):
        activations[name] = output.detach()
    return hook


handles = [
    model.features[7].register_forward_hook(capture("features[7]")),
    model.features[5].register_forward_hook(capture("features[5]")),
]

dummy = torch.zeros(1, 3, 320, 320, device=device)

with torch.no_grad():
    logits = model(dummy)

print("input shape  :", tuple(dummy.shape))
print("logits shape :", tuple(logits.shape))
print()

for name in ("features[5]", "features[7]"):
    shape = tuple(activations[name].shape)
    print(f"{name:<12} activation {shape}   -> spatial {shape[2]}x{shape[3]}, {shape[1]} channels")

for h in handles:
    h.remove()

print()
print("Hooks removed.")

In [ ]:
# ============================================================
# Cell 8 - Real image forward pass
# Confirms the head emits logits and the thresholds line up.
# ============================================================

import pandas as pd
import torchvision.transforms as T
from PIL import Image

eval_transform = T.Compose([
    T.Resize((320, 320)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


def find_nih_root():
    """Locate the dir containing images_001..images_012 without
    descending into the 100k-file image folders."""
    for dirpath, dirnames, filenames in os.walk(ROOT):
        if "images_001" in dirnames:
            return dirpath
        dirnames[:] = [d for d in dirnames if d != "images"]
    return None


NIH_ROOT = find_nih_root()
print("NIH root:", NIH_ROOT)

test_df = pd.read_csv(found["test.csv"][0])
print("test rows:", len(test_df))

index = json.load(open(found["image_index.json"][0]))
filename = test_df["Image Index"].iloc[0]
image_path = os.path.join(NIH_ROOT, index[filename], "images", filename)

print("sample image:", image_path)
print("exists      :", os.path.exists(image_path))

image = Image.open(image_path).convert("RGB")
tensor = eval_transform(image).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(tensor)

probs = torch.sigmoid(logits)[0].cpu().numpy()

print()
print("logits shape :", tuple(logits.shape))
print("logit range  : %.3f .. %.3f" % (logits.min().item(), logits.max().item()))
print("(logits are unbounded -> no sigmoid baked into the classifier)")
print()

thresholds = json.load(open(thr_hits[0])) if thr_hits else {}
truth = test_df[LABELS].iloc[0].values

print(f"{'Pathology':<20}{'prob':>8}{'thresh':>8}{'pred':>7}{'true':>6}")
print("-" * 49)

for i, label in enumerate(LABELS):
    th = thresholds.get(label, 0.5)
    flag = "POS" if probs[i] >= th else "-"
    print(f"{label:<20}{probs[i]:>8.4f}{th:>8.2f}{flag:>7}{int(truth[i]):>6}")

print()
print("=" * 70)
print("ALL VERIFICATION CHECKS PASSED")
print("=" * 70)

In [ ]:
# ============================================================
# Cell 9 - Grad-CAM (Selvaraju et al., 2017)
# Target layer: features[7], confirmed above as the last conv
# stage (10x10x768). One pathology logit at a time, backprop
# from the raw logit (not the sigmoid output) - standard
# Grad-CAM practice, since sigmoid saturates and flattens
# gradients near 0 and 1.
# ============================================================

import torch.nn.functional as Fnn

GRADCAM_LAYER = model.features[7]


def generate_gradcam(model, input_tensor, class_idx, target_layer=GRADCAM_LAYER):
    """Grad-CAM heatmap for one class. Returns (H_in, W_in) numpy array in [0, 1]."""
    activations = {}
    gradients = {}

    def fwd_hook(module, inp, out):
        activations["value"] = out

    def bwd_hook(module, grad_in, grad_out):
        gradients["value"] = grad_out[0]

    h1 = target_layer.register_forward_hook(fwd_hook)
    h2 = target_layer.register_full_backward_hook(bwd_hook)

    model.zero_grad(set_to_none=True)
    out_logits = model(input_tensor)
    score = out_logits[0, class_idx]
    score.backward()

    h1.remove()
    h2.remove()

    acts = activations["value"][0]     # [C, h, w]
    grads = gradients["value"][0]      # [C, h, w]

    # alpha_k = global-average-pooled gradient for channel k (the Grad-CAM weight)
    weights = grads.mean(dim=(1, 2))                    # [C]
    cam = torch.einsum("c,chw->hw", weights, acts)       # weighted sum over channels
    cam = Fnn.relu(cam)                                  # keep only positive evidence

    cam = cam.unsqueeze(0).unsqueeze(0)
    cam = Fnn.interpolate(
        cam, size=input_tensor.shape[-2:], mode="bilinear", align_corners=False
    )
    cam = cam.squeeze().detach().cpu().numpy()

    cam_min, cam_max = cam.min(), cam.max()
    if cam_max - cam_min > 1e-8:
        cam = (cam - cam_min) / (cam_max - cam_min)
    else:
        cam = np.zeros_like(cam)

    return cam, float(out_logits[0, class_idx].item())


# Smoke test on the same sample image used above, explaining its top predicted class.
top_idx = int(np.argmax(probs))
smoke_cam, smoke_logit = generate_gradcam(model, tensor, top_idx)

print("Grad-CAM target layer :", "features[7]")
print(
    "Class explained       :",
    LABELS[top_idx],
    f"(logit={smoke_logit:.3f}, prob={probs[top_idx]:.3f})",
)
print("CAM shape              :", smoke_cam.shape)
print("CAM min/max            : %.3f / %.3f" % (smoke_cam.min(), smoke_cam.max()))

assert smoke_cam.shape == (320, 320)
assert smoke_cam.min() >= 0.0 and smoke_cam.max() <= 1.0 + 1e-6
print("Grad-CAM smoke test passed.")

In [ ]:
# ============================================================
# Cell 10 - Load ground-truth bounding boxes
# Columns are resolved by inspection, not assumed. The classic
# NIH BBox_List_2017.csv header has a well-known quirk: the
# box coordinates are split across several oddly-named columns
# ("Bbox [x", "y", "w", "h]") because of a trailing comma.
# ============================================================

bbox_path = found["BBox_List_2017.csv"][0]
bbox_df = pd.read_csv(bbox_path)
bbox_df.columns = [c.strip() for c in bbox_df.columns]

print("BBox_List_2017.csv columns:", bbox_df.columns.tolist())
print("rows:", len(bbox_df))

col_map = {}
for c in bbox_df.columns:
    lc = c.lower()
    if lc.startswith("image"):
        col_map["image"] = c
    elif "finding" in lc or "label" in lc:
        col_map["label"] = c
    elif lc.startswith("bbox") or lc == "x":
        col_map["x"] = c
    elif lc == "y":
        col_map["y"] = c
    elif lc == "w":
        col_map["w"] = c
    elif lc.startswith("h"):
        col_map["h"] = c

print("Resolved column map:", col_map)

required = ("image", "label", "x", "y", "w", "h")
assert all(k in col_map for k in required), f"Could not resolve bbox columns: {col_map}"

ANNOTATED_CLASSES = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration",
    "Mass", "Nodule", "Pneumonia", "Pneumothorax",
]

print()
print("Boxes per annotated class:")
for cls in ANNOTATED_CLASSES:
    n = (bbox_df[col_map["label"]].str.strip() == cls).sum()
    print(f"  {cls:<20} {n}")

In [ ]:
# ============================================================
# Cell 11 - Grad-CAM overlays on loc_report, one image per
# annotated class, with the ground-truth box drawn for
# comparison.
#
# This is a visual proof of concept only, NOT the IoBB metric.
# Per the project plan: tune only on loc_tune, report only on
# loc_report, and keep classification/localization metrics
# separate. The IoBB threshold sweep is a later, separate step.
# ============================================================

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as cm

loc_report_df = pd.read_csv(found["loc_report.csv"][0])
print("loc_report rows:", len(loc_report_df))

OUT_DIR = "/kaggle/working/gradcam_samples"
os.makedirs(OUT_DIR, exist_ok=True)

loc_report_images = set(loc_report_df["Image Index"])
manifest = []

for cls in ANNOTATED_CLASSES:
    cls_idx = LABELS.index(cls)

    boxes_for_cls = bbox_df[bbox_df[col_map["label"]].str.strip() == cls]
    if boxes_for_cls.empty:
        print(f"[{cls}] no boxes in BBox_List_2017.csv, skipping")
        continue

    candidate_fn, box_row = None, None
    for _, row in boxes_for_cls.iterrows():
        fn = row[col_map["image"]]
        if fn in loc_report_images and fn in index:
            candidate_fn, box_row = fn, row
            break

    if candidate_fn is None:
        print(f"[{cls}] no annotated image found inside loc_report, skipping")
        continue

    img_path = os.path.join(NIH_ROOT, index[candidate_fn], "images", candidate_fn)
    sample_image = Image.open(img_path).convert("RGB")
    orig_w, orig_h = sample_image.size

    input_tensor = eval_transform(sample_image).unsqueeze(0).to(device)
    cam, logit_val = generate_gradcam(model, input_tensor, cls_idx)
    prob = float(torch.sigmoid(torch.tensor(logit_val)))

    resized_image = sample_image.resize((320, 320))

    bx = float(box_row[col_map["x"]]) * (320 / orig_w)
    by = float(box_row[col_map["y"]]) * (320 / orig_h)
    bw = float(box_row[col_map["w"]]) * (320 / orig_w)
    bh = float(box_row[col_map["h"]]) * (320 / orig_h)

    heat = cm.jet(cam)[:, :, :3]
    base = np.asarray(resized_image).astype(np.float32) / 255.0
    overlay = np.clip(0.55 * base + 0.45 * heat, 0, 1)

    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    ax.imshow(overlay)
    ax.add_patch(
        plt.Rectangle((bx, by), bw, bh, fill=False, edgecolor="lime", linewidth=2)
    )
    ax.set_title(f"{cls}  prob={prob:.2f}", fontsize=11)
    ax.axis("off")

    out_path = os.path.join(OUT_DIR, f"gradcam_{cls}.png")
    fig.savefig(out_path, bbox_inches="tight", dpi=120)
    plt.close(fig)

    # Soft sanity check only: fraction of high-activation CAM pixels
    # that fall inside the ground-truth box. This is NOT the official
    # IoBB metric (which needs the 0.1/0.25/0.5 threshold sweep on
    # loc_tune) - it is a cheap directional check that the hot region
    # is roughly in the right place before investing in that sweep.
    hot = cam >= 0.5
    box_mask = np.zeros_like(cam, dtype=bool)
    y0, y1 = int(max(0, by)), int(min(320, by + bh))
    x0, x1 = int(max(0, bx)), int(min(320, bx + bw))
    box_mask[y0:y1, x0:x1] = True
    hot_total = hot.sum()
    frac_in_box = float((hot & box_mask).sum()) / hot_total if hot_total > 0 else float("nan")

    print(
        f"[{cls}] image={candidate_fn}  prob={prob:.3f}  "
        f"hot-pixels-in-box={frac_in_box:.2%}  saved={out_path}"
    )

    manifest.append({
        "pathology": cls,
        "image": candidate_fn,
        "probability": prob,
        "threshold": thresholds.get(cls),
        "positive_prediction": bool(prob >= thresholds.get(cls, 0.5)),
        "gt_box_xywh_320": [bx, by, bw, bh],
        "sanity_hot_pixels_in_box_fraction": (
            None if frac_in_box != frac_in_box else round(frac_in_box, 4)
        ),
        "saved_path": out_path,
    })

with open(os.path.join(OUT_DIR, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)

print()
print(f"Saved {len(manifest)} / {len(ANNOTATED_CLASSES)} overlays to {OUT_DIR}")
print("manifest.json written")

In [ ]:
# ============================================================
# Cell 12 - Summary
# ============================================================

print("=" * 70)
print("GRAD-CAM GENERATION COMPLETE")
print("=" * 70)
print("Target layer     : features[7]  (10x10x768 at 320x320 input)")
print(f"Overlays saved   : {len(manifest)} / {len(ANNOTATED_CLASSES)} annotated classes")
print(f"Output directory : {OUT_DIR}")
for entry in manifest:
    print(f"  - {entry['pathology']:<20} {entry['saved_path']}")

In [ ]:
# ============================================================
# Cell 13 - Localization metrics and box helpers
#
# Two metrics, used for two different jobs:
#
#   IoBB = intersection / ground-truth-box area.  This is what
#   the project reports, because Grad-CAM heatmaps are diffuse
#   and IoU against a small tight box reads as failure even when
#   the model found the right region.
#
#   IoU = intersection / union.  Used ONLY to choose the CAM
#   binarization threshold and the target layer.  IoBB cannot be
#   used for that: its denominator ignores the predicted box, so
#   enlarging the prediction can never lower it, and "maximize
#   mean IoBB" is therefore won by the loosest possible threshold
#   no matter where the heat actually is.  IoU's union term
#   penalizes an oversized box, giving a real interior optimum.
#
# Every reported IoBB number is published next to the mean
# predicted-box area, because IoBB without a size figure is not
# interpretable (a whole-image box scores IoBB = 1.0 every time).
# ============================================================

import scipy.ndimage as ndi
from collections import Counter

IMG_PX = 320
REPORT_THRESHOLDS = (0.10, 0.25, 0.50)
THRESHOLD_GRID = [round(float(t), 2) for t in np.arange(0.05, 0.95, 0.05)]

# Both candidate target layers, confirmed live in Cell 7:
#   features[7] -> 10x10x768 (last conv stage)
#   features[5] -> 20x20x384 (one stage earlier, finer grid)
CAM_LAYERS = {
    "features[5]": model.features[5],
    "features[7]": model.features[7],
}


def cam_to_bbox(cam, T):
    """Bounding box of the largest connected component of {cam >= T}.
    Returns (x0, y0, x1, y1) or None if nothing clears T."""
    mask = cam >= T
    if not mask.any():
        return None
    labeled, n = ndi.label(mask)
    if n == 0:
        return None
    counts = np.bincount(labeled.ravel())
    counts[0] = 0                      # ignore background
    biggest = int(np.argmax(counts))
    ys, xs = np.where(labeled == biggest)
    return (int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1)


def _area(b):
    if b is None:
        return 0.0
    x0, y0, x1, y1 = b
    return max(0.0, x1 - x0) * max(0.0, y1 - y0)


def _inter(a, b):
    ax0, ay0, ax1, ay1 = a
    bx0, by0, bx1, by1 = b
    ix0, iy0 = max(ax0, bx0), max(ay0, by0)
    ix1, iy1 = min(ax1, bx1), min(ay1, by1)
    return max(0.0, ix1 - ix0) * max(0.0, iy1 - iy0)


def iobb(pred_box, gt_box):
    """Intersection over the GROUND-TRUTH box's area."""
    if pred_box is None:
        return 0.0
    return _inter(pred_box, gt_box) / max(1e-6, _area(gt_box))


def iou(pred_box, gt_box):
    """Standard intersection over union."""
    if pred_box is None:
        return 0.0
    inter = _inter(pred_box, gt_box)
    union = _area(pred_box) + _area(gt_box) - inter
    return inter / union if union > 0 else 0.0


def snap_T(T):
    """Map a threshold back onto an exact grid key, so a float that has
    round-tripped through pandas can never raise KeyError mid-run."""
    return min(THRESHOLD_GRID, key=lambda g: abs(g - float(T)))


def best_over_gt(metric_fn, pred_box, gt_boxes):
    """An image can carry several boxes for one class (e.g. two nodules);
    a prediction counts against whichever it matches best."""
    return max((metric_fn(pred_box, gt) for gt in gt_boxes), default=0.0)


def centered_box_like(pred_box):
    """Control baseline: same width/height as the model's box, but parked
    at the image centre. If the model cannot beat this, it is not
    localizing - it is just drawing a blob of about the right size."""
    if pred_box is None:
        return None
    w = pred_box[2] - pred_box[0]
    h = pred_box[3] - pred_box[1]
    c = IMG_PX / 2.0
    return (c - w / 2.0, c - h / 2.0, c + w / 2.0, c + h / 2.0)


def bootstrap_ci(values, stat_fn, n=1000, seed=42):
    """Percentile bootstrap CI, matching the project's classification CIs."""
    vals = np.asarray(values, dtype=float)
    if len(vals) == 0:
        return (float("nan"), float("nan"))
    rng = np.random.default_rng(seed)
    draws = [stat_fn(vals[rng.integers(0, len(vals), len(vals))]) for _ in range(n)]
    return float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5))


print("Metrics defined.")
print("Candidate layers :", list(CAM_LAYERS))
print("Threshold grid   :", THRESHOLD_GRID)
print("Report cutoffs   :", REPORT_THRESHOLDS)

In [ ]:
# ============================================================
# Cell 14 - Multi-layer Grad-CAM in a single backward pass
# Hooking both candidate layers at once means comparing them
# costs one forward/backward per image, not two.
# ============================================================

def gradcam_multi(model, input_tensor, class_idx, layers):
    """Grad-CAM for several layers at once.
    Returns ({layer_name: (H,W) array in [0,1]}, logit, prob)."""
    acts, grads, handles = {}, {}, []

    def fwd(name):
        def hook(module, inp, out):
            acts[name] = out
        return hook

    def bwd(name):
        def hook(module, gin, gout):
            grads[name] = gout[0]
        return hook

    for name, layer in layers.items():
        handles.append(layer.register_forward_hook(fwd(name)))
        handles.append(layer.register_full_backward_hook(bwd(name)))

    model.zero_grad(set_to_none=True)
    out_logits = model(input_tensor)
    score = out_logits[0, class_idx]
    score.backward()

    for h in handles:
        h.remove()

    cams = {}
    for name in layers:
        a = acts[name][0]                      # [C, h, w]
        g = grads[name][0]                     # [C, h, w]
        weights = g.mean(dim=(1, 2))           # alpha_k
        cam = Fnn.relu(torch.einsum("c,chw->hw", weights, a))
        cam = Fnn.interpolate(
            cam[None, None], size=input_tensor.shape[-2:],
            mode="bilinear", align_corners=False,
        ).squeeze().detach().cpu().numpy()
        lo, hi = float(cam.min()), float(cam.max())
        cams[name] = (cam - lo) / (hi - lo) if (hi - lo) > 1e-8 else np.zeros_like(cam)

    logit = float(out_logits[0, class_idx].item())
    prob = float(torch.sigmoid(out_logits[0, class_idx]).item())
    return cams, logit, prob


# Sanity check against the single-layer implementation from Cell 9.
_chk_cams, _chk_logit, _chk_prob = gradcam_multi(model, tensor, top_idx, CAM_LAYERS)
_max_delta = float(np.abs(_chk_cams["features[7]"] - smoke_cam).max())
print("multi-layer vs single-layer features[7] max |diff| :", f"{_max_delta:.2e}")
assert _max_delta < 1e-3, "multi-layer Grad-CAM disagrees with the Cell 9 implementation"
for _n, _c in _chk_cams.items():
    assert _c.shape == (IMG_PX, IMG_PX)
    assert _c.min() >= 0.0 and _c.max() <= 1.0 + 1e-6
print("gradcam_multi agrees with generate_gradcam. OK.")

In [ ]:
# ============================================================
# Cell 15 - Build annotated (image, class, boxes) pairs
# One image can hold several boxes for the same class, so all of
# them are kept per pair rather than only the first.
# ============================================================

loc_tune_df = pd.read_csv(found["loc_tune.csv"][0])
loc_tune_images = set(loc_tune_df["Image Index"])
print("loc_tune rows:", len(loc_tune_df))

# Patient-level separation is the whole point of these two splits;
# verify it here rather than trusting it.
if "Patient ID" in loc_tune_df.columns and "Patient ID" in loc_report_df.columns:
    _tune_pids = set(loc_tune_df["Patient ID"])
    _report_pids = set(loc_report_df["Patient ID"])
    print("loc_tune patients:", len(_tune_pids), "| loc_report patients:", len(_report_pids))
    print("shared patients  :", len(_tune_pids & _report_pids))
    assert not (_tune_pids & _report_pids), "loc_tune and loc_report share patients"
else:
    print("WARNING: no 'Patient ID' column; skipping patient-overlap check.")
    print("columns:", loc_tune_df.columns.tolist())


def build_pairs(split_image_set, classes):
    pairs = []
    sub = bbox_df[bbox_df[col_map["label"]].str.strip().isin(classes)]
    for (fn, raw_cls), group in sub.groupby([col_map["image"], col_map["label"]]):
        cls = raw_cls.strip()
        if fn not in split_image_set or fn not in index:
            continue
        img_path = os.path.join(NIH_ROOT, index[fn], "images", fn)
        if not os.path.exists(img_path):
            continue
        with Image.open(img_path) as im:
            ow, oh = im.size
        boxes = []
        for _, row in group.iterrows():
            bx = float(row[col_map["x"]]) * (IMG_PX / ow)
            by = float(row[col_map["y"]]) * (IMG_PX / oh)
            bw = float(row[col_map["w"]]) * (IMG_PX / ow)
            bh = float(row[col_map["h"]]) * (IMG_PX / oh)
            boxes.append((bx, by, bx + bw, by + bh))
        pairs.append((fn, cls, boxes))
    return pairs


tune_pairs = build_pairs(loc_tune_images, ANNOTATED_CLASSES)
report_pairs = build_pairs(loc_report_images, ANNOTATED_CLASSES)

print("tune_pairs   :", len(tune_pairs))
print("report_pairs :", len(report_pairs))
print("tune by class   :", dict(sorted(Counter(c for _, c, _ in tune_pairs).items())))
print("report by class :", dict(sorted(Counter(c for _, c, _ in report_pairs).items())))

_overlap_imgs = set(f for f, _, _ in tune_pairs) & set(f for f, _, _ in report_pairs)
print("images in both splits:", len(_overlap_imgs))
assert not _overlap_imgs, "an image appears in both loc_tune and loc_report"

In [ ]:
# ============================================================
# Cell 16 - One Grad-CAM per pair, boxes cached per (layer, T)
# The CAM is computed once; thresholding is cheap and done for
# the whole grid up front, so the sweep never re-runs the model.
# ============================================================

def compute_records(pairs, tag):
    records = []
    for i, (fn, cls, gt_boxes) in enumerate(pairs):
        img_path = os.path.join(NIH_ROOT, index[fn], "images", fn)
        pil_img = Image.open(img_path).convert("RGB")
        x = eval_transform(pil_img).unsqueeze(0).to(device)
        cams, logit, prob = gradcam_multi(model, x, LABELS.index(cls), CAM_LAYERS)
        records.append({
            "image": fn,
            "pathology": cls,
            "gt_boxes": gt_boxes,
            "logit": logit,
            "prob": prob,
            "boxes": {
                lname: {T: cam_to_bbox(cam, T) for T in THRESHOLD_GRID}
                for lname, cam in cams.items()
            },
        })
        if (i + 1) % 50 == 0 or (i + 1) == len(pairs):
            print(f"[{tag}] {i + 1}/{len(pairs)} pairs processed")
    return records


tune_records = compute_records(tune_pairs, "tune")
report_records = compute_records(report_pairs, "report")
print("Done. tune:", len(tune_records), "| report:", len(report_records))

In [ ]:
# ============================================================
# Cell 17 - Select target layer and threshold, on loc_tune only
# Criterion: mean IoU (well-posed). mean_iobb and box area are
# printed alongside so the degeneracy is visible rather than
# hidden: watch mean_iobb rise as T falls, all the way down.
# ============================================================

def sweep_layer(records, layer):
    rows = []
    for T in THRESHOLD_GRID:
        ious, iobbs, areas = [], [], []
        for r in records:
            pb = r["boxes"][layer][snap_T(T)]
            ious.append(best_over_gt(iou, pb, r["gt_boxes"]))
            iobbs.append(best_over_gt(iobb, pb, r["gt_boxes"]))
            areas.append(_area(pb) / (IMG_PX ** 2))
        iobbs_arr = np.array(iobbs)
        row = {
            "layer": layer,
            "T": T,
            "mean_iou": float(np.mean(ious)),
            "mean_iobb": float(iobbs_arr.mean()),
            "box_area_frac": float(np.mean(areas)),
        }
        for tau in REPORT_THRESHOLDS:
            row[f"iobb_acc@{tau}"] = float((iobbs_arr > tau).mean())
        rows.append(row)
    return pd.DataFrame(rows)


tune_sweep = pd.concat(
    [sweep_layer(tune_records, l) for l in CAM_LAYERS], ignore_index=True
)
for layer in CAM_LAYERS:
    print()
    print(f"--- loc_tune sweep, {layer} ---")
    print(tune_sweep[tune_sweep["layer"] == layer].to_string(index=False))

print()
print("Note the mean_iobb column: it rises monotonically as T falls, and")
print("box_area_frac rises with it. That is why the threshold is chosen on")
print("mean_iou, which trades off coverage against box size.")

best_row = tune_sweep.loc[tune_sweep["mean_iou"].idxmax()]
BEST_LAYER = str(best_row["layer"])
T_STAR = float(best_row["T"])

print()
print("=" * 62)
print(f"Selected layer     : {BEST_LAYER}")
print(f"Selected threshold : T* = {T_STAR}")
print(f"  loc_tune mean IoU      : {best_row['mean_iou']:.4f}")
print(f"  loc_tune mean IoBB     : {best_row['mean_iobb']:.4f}")
print(f"  loc_tune box area frac : {best_row['box_area_frac']:.4f}")
print("=" * 62)

layer_summary = (
    tune_sweep.groupby("layer")["mean_iou"].max().rename("best_mean_iou").reset_index()
)
print()
print("Best achievable mean IoU per layer (loc_tune):")
print(layer_summary.to_string(index=False))

# Per-class thresholds, for comparison against the single global T*.
per_class_T = {}
for cls in ANNOTATED_CLASSES:
    subset = [r for r in tune_records if r["pathology"] == cls]
    if not subset:
        continue
    best_T, best_score = None, -1.0
    for T in THRESHOLD_GRID:
        score = float(np.mean([
            best_over_gt(iou, r["boxes"][BEST_LAYER][snap_T(T)], r["gt_boxes"]) for r in subset
        ]))
        if score > best_score:
            best_T, best_score = T, score
    per_class_T[cls] = best_T

print()
print("Per-class thresholds tuned on loc_tune (small n each - overfitting risk):")
for cls, T in sorted(per_class_T.items()):
    n = sum(1 for r in tune_records if r["pathology"] == cls)
    print(f"  {cls:<20} T={T:.2f}  (n={n})")

In [ ]:
# ============================================================
# Cell 18 - Report on loc_report, which tuning never touched
#
# Reported together, deliberately:
#   - IoBB accuracy at 0.10 / 0.25 / 0.50 (the project's metric)
#   - mean IoU and mean predicted-box area (so the IoBB numbers
#     can be read honestly)
#   - a centred-box control of identical size per image
#   - the whole-image reference, which scores IoBB = 1.0 always
# ============================================================

def evaluate(records, layer, threshold_fn, tag):
    rows = []
    for r in records:
        T = threshold_fn(r["pathology"])
        pb = r["boxes"][layer][snap_T(T)]
        ctrl = centered_box_like(pb)
        rows.append({
            "image": r["image"],
            "pathology": r["pathology"],
            "T": T,
            "prob": r["prob"],
            "iobb": best_over_gt(iobb, pb, r["gt_boxes"]),
            "iou": best_over_gt(iou, pb, r["gt_boxes"]),
            "box_area_frac": _area(pb) / (IMG_PX ** 2),
            "iobb_centered_ctrl": best_over_gt(iobb, ctrl, r["gt_boxes"]),
            "n_gt_boxes": len(r["gt_boxes"]),
        })
    return pd.DataFrame(rows)


def summarize(df, tag):
    out = []
    for cls in ANNOTATED_CLASSES:
        sub = df[df["pathology"] == cls]
        if sub.empty:
            continue
        row = {
            "pathology": cls,
            "n_images": int(len(sub)),
            "mean_iobb": float(sub["iobb"].mean()),
            "mean_iou": float(sub["iou"].mean()),
            "box_area_frac": float(sub["box_area_frac"].mean()),
        }
        for tau in REPORT_THRESHOLDS:
            acc = float((sub["iobb"] > tau).mean())
            lo, hi = bootstrap_ci(
                (sub["iobb"] > tau).astype(float).values, lambda v: float(v.mean())
            )
            row[f"iobb_acc@{tau}"] = acc
            row[f"iobb_acc@{tau}_lo"] = lo
            row[f"iobb_acc@{tau}_hi"] = hi
        row["ctrl_acc@0.25"] = float((sub["iobb_centered_ctrl"] > 0.25).mean())
        out.append(row)
    res = pd.DataFrame(out)
    macro = {
        "pathology": "MACRO (class-avg)",
        "n_images": int(len(df)),
        "mean_iobb": float(res["mean_iobb"].mean()),
        "mean_iou": float(res["mean_iou"].mean()),
        "box_area_frac": float(res["box_area_frac"].mean()),
    }
    for tau in REPORT_THRESHOLDS:
        macro[f"iobb_acc@{tau}"] = float(res[f"iobb_acc@{tau}"].mean())
        macro[f"iobb_acc@{tau}_lo"] = float("nan")
        macro[f"iobb_acc@{tau}_hi"] = float("nan")
    macro["ctrl_acc@0.25"] = float(res["ctrl_acc@0.25"].mean())
    return res, pd.DataFrame([macro])


report_global = evaluate(report_records, BEST_LAYER, lambda c: T_STAR, "global")
report_perclass = evaluate(
    report_records, BEST_LAYER, lambda c: per_class_T.get(c, T_STAR), "perclass"
)

res_global, macro_global = summarize(report_global, "global")
res_perclass, macro_perclass = summarize(report_perclass, "perclass")

show_cols = [
    "pathology", "n_images", "mean_iobb", "mean_iou", "box_area_frac",
    "iobb_acc@0.1", "iobb_acc@0.25", "iobb_acc@0.5", "ctrl_acc@0.25",
]

print("=" * 78)
print(f"loc_report - {BEST_LAYER}, single global T* = {T_STAR}")
print("=" * 78)
print(res_global[show_cols].round(4).to_string(index=False))
print()
print(macro_global[show_cols].round(4).to_string(index=False))

print()
print("95% CI on IoBB accuracy @0.25 (1000 bootstrap draws, per class):")
for _, r in res_global.iterrows():
    print(
        f"  {r['pathology']:<20} {r['iobb_acc@0.25']:.3f}  "
        f"[{r['iobb_acc@0.25_lo']:.3f}, {r['iobb_acc@0.25_hi']:.3f}]  (n={int(r['n_images'])})"
    )

print()
print("=" * 78)
print(f"loc_report - {BEST_LAYER}, per-class thresholds")
print("=" * 78)
print(res_perclass[show_cols].round(4).to_string(index=False))
print()
print(macro_perclass[show_cols].round(4).to_string(index=False))

# Whole-image reference: the reason IoBB is never reported without box size.
whole = (0, 0, IMG_PX, IMG_PX)
whole_iobb = np.array([best_over_gt(iobb, whole, r["gt_boxes"]) for r in report_records])
whole_iou = np.array([best_over_gt(iou, whole, r["gt_boxes"]) for r in report_records])
print()
print("Degenerate reference - predict the ENTIRE image every time:")
print(f"  mean IoBB           : {whole_iobb.mean():.4f}")
print(f"  IoBB acc @0.25      : {(whole_iobb > 0.25).mean():.4f}")
print(f"  mean IoU            : {whole_iou.mean():.4f}   <- IoU exposes it, IoBB does not")
print(f"  box area fraction   : 1.0000")

results_path = os.path.join(OUT_DIR, "gradcam_iobb_results.csv")
pd.concat([res_global, macro_global], ignore_index=True).to_csv(results_path, index=False)
pd.concat([res_perclass, macro_perclass], ignore_index=True).to_csv(
    os.path.join(OUT_DIR, "gradcam_iobb_results_perclass_T.csv"), index=False
)
report_global.to_csv(os.path.join(OUT_DIR, "gradcam_iobb_per_image.csv"), index=False)
tune_sweep.to_csv(os.path.join(OUT_DIR, "gradcam_tune_sweep.csv"), index=False)

with open(os.path.join(OUT_DIR, "gradcam_iobb_config.json"), "w") as f:
    json.dump({
        "target_layer": BEST_LAYER,
        "layer_candidates": list(CAM_LAYERS),
        "cam_threshold_T_star": T_STAR,
        "per_class_thresholds": per_class_T,
        "selection_criterion": "argmax mean IoU on loc_tune (NOT mean IoBB - degenerate)",
        "reported_metric": "IoBB",
        "report_thresholds": list(REPORT_THRESHOLDS),
        "tuned_on": "loc_tune",
        "reported_on": "loc_report",
        "n_tune_pairs": len(tune_pairs),
        "n_report_pairs": len(report_pairs),
    }, f, indent=2)

print()
print("Wrote gradcam_iobb_results.csv, gradcam_iobb_results_perclass_T.csv,")
print("      gradcam_iobb_per_image.csv, gradcam_tune_sweep.csv, gradcam_iobb_config.json")

In [ ]:
# ============================================================
# Cell 19 - Visual spot check: best / median / worst case
# green = ground truth, red dashed = predicted box at T*
# ============================================================

ordered = report_global.sort_values("iobb", ascending=False).reset_index(drop=True)
picks = [
    ("best", ordered.iloc[0]),
    ("median", ordered.iloc[len(ordered) // 2]),
    ("worst", ordered.iloc[-1]),
]

for tag, row in picks:
    fn, cls = row["image"], row["pathology"]
    rec = next(r for r in report_records if r["image"] == fn and r["pathology"] == cls)

    pil_img = Image.open(os.path.join(NIH_ROOT, index[fn], "images", fn)).convert("RGB")
    x = eval_transform(pil_img).unsqueeze(0).to(device)
    cams, _, prob = gradcam_multi(model, x, LABELS.index(cls), CAM_LAYERS)
    cam = cams[BEST_LAYER]
    pred = cam_to_bbox(cam, row["T"])

    base = np.asarray(pil_img.resize((IMG_PX, IMG_PX))).astype(np.float32) / 255.0
    overlay = np.clip(0.55 * base + 0.45 * cm.jet(cam)[:, :, :3], 0, 1)

    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    ax.imshow(overlay)
    for gx0, gy0, gx1, gy1 in rec["gt_boxes"]:
        ax.add_patch(plt.Rectangle(
            (gx0, gy0), gx1 - gx0, gy1 - gy0,
            fill=False, edgecolor="lime", linewidth=2,
        ))
    if pred is not None:
        px0, py0, px1, py1 = pred
        ax.add_patch(plt.Rectangle(
            (px0, py0), px1 - px0, py1 - py0,
            fill=False, edgecolor="red", linewidth=2, linestyle="--",
        ))
    ax.set_title(
        f"{tag}: {cls}  IoBB={row['iobb']:.2f}  IoU={row['iou']:.2f}  p={prob:.2f}",
        fontsize=10,
    )
    ax.axis("off")

    out_path = os.path.join(OUT_DIR, f"iobb_{tag}_{cls}_{fn.replace('.png', '')}.png")
    fig.savefig(out_path, bbox_inches="tight", dpi=120)
    plt.close(fig)
    print(f"saved [{tag}] {cls} IoBB={row['iobb']:.3f} -> {out_path}")

In [ ]:
# ============================================================
# Cell 20 - Final summary
# ============================================================

print("=" * 78)
print("GRAD-CAM + IoBB LOCALIZATION EVALUATION COMPLETE")
print("=" * 78)
print(f"Target layer            : {BEST_LAYER}  (chosen over {list(CAM_LAYERS)} by mean IoU on loc_tune)")
print(f"CAM threshold T*        : {T_STAR}  (mean IoU criterion)")
print(f"Tuned on                : loc_tune   ({len(tune_pairs)} annotated image-class pairs)")
print(f"Reported on             : loc_report ({len(report_pairs)} annotated image-class pairs)")
print(f"Classes evaluated       : {len(res_global)} of {len(ANNOTATED_CLASSES)} annotated classes")
print()
print(macro_global[show_cols].round(4).to_string(index=False))
print()
print("Read the IoBB columns next to box_area_frac and ctrl_acc@0.25:")
print("  box_area_frac  - how much of the image the prediction covers")
print("  ctrl_acc@0.25  - same-sized box parked at the image centre")
print("A score that does not beat the centred control is not localization.")